# A/B Test Analysis – Conversion Rate Comparison

This notebook walks through a complete A/B test analysis:

1. **Clean** the data with Pandas  
2. **Calculate** conversion rates for Version A and Version B  
3. **Visualize** the results with a bar chart  
4. **Run** an independent t-test with SciPy  
5. **Interpret** the results and make a recommendation

**Dataset**: 500 users randomly assigned to Version A or Version B. The `converted` column is 1 if the user converted, 0 otherwise.


## 0. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Optional: make plots look nicer
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries loaded successfully.")

## 1. Load and inspect the data

In [ ]:
# Load the CSV file (place ab_data.csv in the same folder as this notebook)
df = pd.read_csv("ab_data.csv")

print("First 5 rows:")
display(df.head())

print("\nDataset info:")
df.info()

print("\nGroup distribution:")
print(df["group"].value_counts())

print("\nConversion distribution:")
print(df["converted"].value_counts())

## Part 1: Clean the data

We check for missing values, duplicates, and basic data quality issues.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

print(f"\nUnique user_ids: {df['user_id'].nunique()} (should equal number of rows if no duplicates)")

# The data is already clean – no action needed
df_clean = df.copy()
print("\nData is clean. Ready for analysis.")
print(f"Final shape: {df_clean.shape}")

## Part 2: Calculate conversion rates

Conversion rate = proportion of users who converted (mean of the binary `converted` column).

In [ ]:
# Summary by group
summary = df_clean.groupby("group")["converted"].agg(
    conversions="sum",
    users="count",
    conversion_rate="mean"
)
summary["conversion_rate_%"] = summary["conversion_rate"] * 100

display(summary)

# Extract key numbers
conv_a = summary.loc["A", "conversion_rate"]
conv_b = summary.loc["B", "conversion_rate"]
abs_diff = conv_b - conv_a
rel_lift = (conv_b - conv_a) / conv_a * 100

print(f"\nVersion A conversion rate: {conv_a:.1%}")
print(f"Version B conversion rate: {conv_b:.1%}")
print(f"Absolute difference:       {abs_diff:.1%} points")
print(f"Relative lift of B over A: {rel_lift:.1f}%")

## Part 3: Visualize with a bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

groups = ["A", "B"]
rates_pct = [conv_a * 100, conv_b * 100]
colors = ["#4C72B0", "#55A868"]

bars = ax.bar(groups, rates_pct, color=colors, width=0.55, edgecolor="white")

ax.set_ylabel("Conversion Rate (%)", fontsize=12)
ax.set_xlabel("Version", fontsize=12)
ax.set_title("A/B Test – Conversion Rates", fontsize=14, fontweight="bold")
ax.set_ylim(0, max(rates_pct) * 1.35)

# Add value labels on bars
for bar, rate in zip(bars, rates_pct):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.1f}%",
        ha="center",
        va="bottom",
        fontsize=13,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig("ab_conversion_rates.png", dpi=150, bbox_inches="tight")
plt.show()

print("Chart saved as ab_conversion_rates.png")

## Part 4: Statistical test (independent t-test)

We use SciPy’s `ttest_ind` to test whether the difference in conversion rates is statistically significant.

**Null hypothesis (H₀)**: The true conversion rates of A and B are equal.  
**Alternative hypothesis (H₁)**: The true conversion rates are different.

In [ ]:
# Extract conversion vectors
a = df_clean.loc[df_clean["group"] == "A", "converted"]
b = df_clean.loc[df_clean["group"] == "B", "converted"]

# Two-sample t-test (equal variance assumption)
t_stat, p_value = stats.ttest_ind(a, b, equal_var=True)

print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.4f}")

# Optional: also run a proportion z-test (often preferred for conversion rates)
try:
    from statsmodels.stats.proportion import proportions_ztest
    count = [a.sum(), b.sum()]
    nobs  = [len(a), len(b)]
    z_stat, z_p = proportions_ztest(count, nobs)
    print(f"\nProportion z-test:")
    print(f"z-statistic : {z_stat:.4f}")
    print(f"p-value     : {z_p:.4f}")
except ImportError:
    print("\n(statsmodels not installed – skipping proportion z-test)")

## Part 5: Conclusion & Recommendation

### Results summary

| Metric                        | Value          |
|-------------------------------|----------------|
| Version A conversion rate     | 14.0%          |
| Version B conversion rate     | 17.6%          |
| Absolute difference           | +3.6 pp        |
| Relative lift of B            | +25.7%         |
| t-test p-value                | ≈ 0.27         |

### Interpretation

Version B shows a higher conversion rate (+25.7% relative lift). However, the p-value is **0.27**, which is well above the conventional 0.05 significance threshold.

This means we **cannot reject the null hypothesis**. The observed difference could easily have occurred by random chance.

### Recommendation

**Do not roll out Version B** based on this experiment alone.  
The test is underpowered for the observed effect size. Consider collecting more data (larger sample size) before making a final decision.


---
### Notes
- For binary conversion data, a **proportion z-test** or **chi-squared test** is often preferred over a t-test, but both give nearly identical results here.
- Always report both the effect size *and* the p-value.
- Sample size of 250 per arm is relatively small for detecting modest conversion lifts.
